# Batch Preprocessing

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 22, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 5 Sonnet  

---

This notebook runs the preprocessing pipeline for all subjects that should be included in the final analyses. It does so by defining a preprocessing function, that takes either a single or multiple subject IDs aswell as subject-specific configuration settings, as inputs.

### 1. Setup: Loading modules and objects, creating basic paths and helper function

This initialization cell prepares the workspace by loading the relevant modules and specifying the relevant paths and directories.

In [1]:
# ==========================================
# 1. IMPORTS & GLOBAL PATHS
# ==========================================
import mne
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
from autoreject import AutoReject
import matplotlib
matplotlib.use('Agg')  # No interactive pop-ups during batch runs
import matplotlib.pyplot as plt
mne.viz.set_browser_backend("matplotlib")

# ==========================================
# 2. Path Definitions & Global Variables 
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 3. Load Configuration dataframe
# ==========================================
df_config = pd.read_csv(config_path, dtype={'subject': str})

# ==========================================
# 3. HELPER FUNCTION: SAVE A FIGURE SAFELY
# ==========================================
def save_fig(fig, out_dir, fname):
    """Save and close a figure, creating the directory if needed."""
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / fname, dpi=100, bbox_inches='tight')
    plt.close(fig)

Using matplotlib as 2D backend.


### 2. Define Preprocessing Pipeline as a customizable function

The preprocessing steps that are included and their order is mainly informed by the [**COBIDAS report** on Best Practices in Data Analysis and Sharing in Neuroimaging using MEEG (Pernet et al., 2018)](https://doi.org/10.1038/s41593-020-00709-0). We will adjust the sequence of steps based on some practical considerations, e.g., we will use the **Autoreject function** as a tool for automated artifact rejection and correction (Jas et al., 2017) to ensure data-driven and reproducible artifact rejection in a semi-automated preprocessing pipeline.



__Preprocessing sequence__ <br>
1. **Bad channel interpolation** goes first because noisy channels would contaminate all subsequent processing steps.
2. **Filtering (high-pass, low-pass)** is a pre-requisite for fitting ICA. By doing it before epoching, we avoid edge artifacts.
3. **Downsampling** reduces computational costs of ICA and Autoreject.
4. **Artefact correction with ICA** requires continuous data to identify stable components.
5. **Epoching (separately for encoding and retrieval)** is performed on clean continuous signal.
6. **Merge epochs & behavioral data frames** so that trial-level behavioral metadata is kept intact before any epochs get dropped.
7. **Autoreject** runs on epochs (not continuous data) to determine optimal peak-to-peak thresholds for the epochs. 
8. **Baseline correction** runs after Autoreject to ensure that bad channels and trials do not bias the baseline estimate.
9. **Re-referencing to electrode average** is done last, because it would be affected by any bad channels or epochs present in the data.

In [ ]:
# ==========================================
# 2. MAIN PREPROCESSING FUNCTION
# ==========================================
def preprocess_subject(
    subject_id,
    bad_channels,
    bad_ics,
    l_freq=0.1, # This is in line with Gian's feedback (calculate ICA with 1 Hz HPF, apply to 0.1 HPF dataset)
    h_freq=40.0,
    resample_sfreq=250.0,
    epoch_tmin=-0.5,
    epoch_tmax=1.0,
    baseline_corr=(None, 0),
    buffer_sec = 20.0,
    overwrite=True,
):
    """
    Run the full preprocessing pipeline for one subject.

    Parameters
    ----------
    subject_id : str
        BIDS subject label, e.g. "23".
    bad_channels : list of str
        Channels to mark as bad and interpolate, from notebook 2's QC.
    bad_ics : list of int
        ICA component indices to exclude, from notebook 2's QC.
    overwrite : bool
        If False and output files already exist, skip this subject
        (idempotent re-runs — safe to re-execute the batch loop).

    Returns
    -------
    dict
        Summary log (also saved as JSON) with key metrics for this subject.
    """

    # Create various file paths
    subj_deriv_dir = derivatives_dir / f"sub-{subject_id}" / "eeg"
    plots_dir = subj_deriv_dir / "qc_plots"
    log_path = subj_deriv_dir / f"sub-{subject_id}_preproc_log.json"
    beh_path = bids_root / f"sub-{subject_id}" / "beh" / f"sub-{subject_id}_task-loc_label-merged_beh.csv"
    epochs_enc_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-encoding_epo.fif"
    epochs_ret_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-retrieval_epo.fif"

    # --- Idempotency check: skip subject if preprocessing was already done ---
    if not overwrite and epochs_enc_path.exists() and epochs_ret_path.exists():
        print(f"sub-{subject_id}: outputs already exist, skipping (overwrite=False).")
        with open(log_path, 'r') as f:
            return json.load(f)

    log = {"subject": subject_id, "steps_completed": [], "warnings": []}

    # ==========================================
    # STEP 1: LOAD RAW + APPLY BAD CHANNELS
    # ==========================================
    bids_path = BIDSPath(subject=subject_id, task='loc', datatype='eeg', root=bids_root)
    raw = read_raw_bids(bids_path=bids_path, verbose='error')
    raw.load_data()
    raw.info['bads'] = bad_channels
    log["bad_channels"] = bad_channels
    log["steps_completed"].append("load_raw")

    # ==========================================
    # STEP 1b: CROP TO TASK-RELEVANT SEGMENTS (encoding + retrieval)
    # ==========================================
    # Drops irrelevant recorded phases (e.g., instructions, breaks, ...).
    # Encoding and retrieval are cropped as separate continuous chunks (each with a buffer) 
    # and concatenated BEFORE filtering — same buffer size as used in notebook 2, 
    # so the QC inspection and the actual preprocessing operate on matching data.

    def get_onset(raw, description):
        onset = raw.annotations.onset[raw.annotations.description == description]
        if len(onset) == 0:
            raise ValueError(f"No annotation '{description}' found for sub-{subject_id}")
        return onset[0]

    enc_start = get_onset(raw, "enc_start")
    enc_end   = get_onset(raw, "enc_end")
    ret_start = get_onset(raw, "ret_start")
    ret_end   = get_onset(raw, "ret_end")

    t_min, t_max = raw.times[0], raw.times[-1]
    enc_tmin = max(enc_start - buffer_sec, t_min)
    enc_tmax = min(enc_end + buffer_sec, t_max)
    ret_tmin = max(ret_start - buffer_sec, t_min)
    ret_tmax = min(ret_end + buffer_sec, t_max)

    raw_enc = raw.copy().crop(tmin=enc_tmin, tmax=enc_tmax)
    raw_ret = raw.copy().crop(tmin=ret_tmin, tmax=ret_tmax)
    raw = mne.concatenate_raws([raw_enc, raw_ret])  # inserts boundary annotation at the seam

    log["crop_params"] = {
        "buffer_sec": buffer_sec,
        "encoding_window": [enc_tmin, enc_tmax],
        "retrieval_window": [ret_tmin, ret_tmax],
        "total_duration_sec": raw.times[-1],
    }
    log["steps_completed"].append("crop_concatenate")


    # ==========================================
    # STEP 2: FILTERING (low, high)
    # ==========================================

    # Plot PSD before filtering
    psd_before = raw.copy().pick('eeg').compute_psd(fmax=80)

    # Apply high pass and low pass filters
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    log["filter_params"] = {"l_freq": l_freq, "h_freq": h_freq}
    log["steps_completed"].append("filter")

    # Plot PSD after filtering
    psd_after = raw.copy().pick('eeg').compute_psd(fmax=60)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    psd_before.plot(axes=axes[0], show=False)
    axes[0].set_title("Before filter")
    psd_after.plot(axes=axes[1], show=False)
    axes[1].set_title("After filter")
    save_fig(fig, plots_dir, "01_psd_before_after_filter.png")


    # Interpolate bad channels now (before ICA apply / epoching)
    if bad_channels:
        raw.interpolate_bads(reset_bads=True, verbose=False)
        log["steps_completed"].append("interpolate_bads")

    # ==========================================
    # STEP 3: DOWNSAMPLE
    # ==========================================
    raw.resample(resample_sfreq, verbose=False)
    log["resample_sfreq"] = resample_sfreq
    log["steps_completed"].append("resample")

    # ==========================================
    # STEP 4: APPLY ICA (using bad_ics from notebook 2)
    # ==========================================
    # #ICA.apply() needs the actual unmixing matrix, which was saved in the previous notebook
    ica_path = derivatives_dir / f"sub-{subject_id}" / "eeg" / "ica_qc" / f"sub-{subject_id}_ica.fif"
    if not ica_path.exists():
        raise FileNotFoundError(
            f"No saved ICA solution found for sub-{subject_id} at {ica_path}. "
            f"Save ica.save(...) at the end of notebook 2."
        )
    ica = mne.preprocessing.read_ica(ica_path)
    ica.exclude = bad_ics
    ica.apply(raw)
    log["excluded_ics"] = bad_ics
    log["steps_completed"].append("ica_apply")
   
    # ==========================================
    # STEP 5: EPOCHING (encoding + retrieval, separately)
    # ==========================================

    # --- Load and prepare behavioral data (encoding and retrival are merged already) ---
    beh_path = (bids_root / f"sub-{subject_id}" / "beh"
                / f"sub-{subject_id}_task-loc_label-merged_beh.csv")
    beh_df = pd.read_csv(beh_path)

    # This function assigns each trial a label for the schema and perceptual condition (either SC or SI, and PC or PI)
    def sc_si_pc_pi(row):
        sc_si = 'SI' if row['enc_high_prediction'] == 1 else 'SC'
        pc_pi = 'PI' if row['enc_low_prediction'] == 1 else 'PC'
        return sc_si, pc_pi

    # We create a dataframe that only contains the 64 encoding trials
    beh_enc = (beh_df.dropna(subset=['enc_trial_count']) # Drops rows that have NULL values in column 'enc_trial_count'
                      .sort_values('enc_trial_count') # Sort encoding trials by trial_count
                      .reset_index(drop=True)) # Discard old indices and sets new 0-index ascending by trial_count

    # Function to assign each row of the encoding data a condition label
    # in the form of < "Target" / "SC" or "SI" / "PC" or "PI" >
    def enc_label(r):
        return 'Target/' + '/'.join(sc_si_pc_pi(r))

    beh_enc['condition_label'] = beh_enc.apply(enc_label, axis=1)

    # Similarly, we create a retrieval dataframe containing 84 rows, in their retrieval presentation order
    beh_ret = beh_df.sort_values('ret_trial_count').reset_index(drop=True)

    def ret_label(row):
        if row['ret_trial_type'] == 'new':
            return 'Cue/New'
        return 'Cue/Old/' + '/'.join(sc_si_pc_pi(row))
    beh_ret['condition_label'] = beh_ret.apply(ret_label, axis=1)

    # Store the behavioral data for the two phases in a dictionairy
    beh_metadata = {'encoding': beh_enc, 'retrieval': beh_ret}

    # --- Extract events from annotations ---
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    # Get unique event codes actually present in this recording 
    present_ids = set(events[:, 2])

    # If we want to include the participants with the behavioral bug, we'd need to add a 'bug' condition here 
    enc_target_event_ids = {
        'Target/SC/PC': event_id['tgt_sc_pc'],
        'Target/SC/PI': event_id['tgt_sc_pi'],
        'Target/SI/PC': event_id['tgt_si_pc'],
        'Target/SI/PI': event_id['tgt_si_pi'],
    }

    ret_cue_event_ids = {
        'Cue/Old/SC/PC': event_id['ret_cue_old_sc_pc'],
        'Cue/Old/SC/PI': event_id['ret_cue_old_sc_pi'],
        'Cue/Old/SI/PC': event_id['ret_cue_old_si_pc'],
        'Cue/Old/SI/PI': event_id['ret_cue_old_si_pi'],
        'Cue/New': event_id['ret_cue_new'],
    }

    # Collect epochs by iterating twice: once with phase 'encoding', once with phase ' retrieval'
    epochs_dict = {}
    for phase, phase_event_ids in [('encoding', enc_target_event_ids),
                                    ('retrieval', ret_cue_event_ids)]:

        # Dict comprehension: we store every pair of label and event ID (label, eid) in phase_event_ids as a new dict entry IF
        # that eid is in present_id.
        available = {label: eid for label, eid in phase_event_ids.items()
                     if eid in present_ids}

        # Check for missing event IDs in a phase
        missing = set(phase_event_ids) - set(available)
        if missing:
            log["warnings"].append(
                f"{phase}: missing conditions in recording — {sorted(missing)}"
            )

        # Check if a phase is wholly missing
        if not available:
            log["warnings"].append(f"No events found for {phase} phase — skipped.")
            continue

        # Filter the full events array down to just this phase's matching triggers,
        # in chronological order — this is what metadata needs to align against.
        phase_mask = np.isin(events[:, 2], list(available.values())) # Boolean masking: vector with length of nrows[events], TRUE if the event code
        # is in our dictionairy of event codes we care about (available)
        phase_events = events[phase_mask] # This contains only encoding (or retrieval events) in original chronological order

        metadata = beh_metadata[phase]

        # Comparing the length of behavioral data to the length of extracted events for a given phase.
        # This check currently fails the subjects with the bug in the task code.
        if len(metadata) != len(phase_events):
            log["warnings"].append(
                f"{phase}: behavioral rows ({len(metadata)}) != triggered EEG events "
                f"({len(phase_events)}) — metadata NOT attached for sub-{subject_id}, check alignment."
            )
            metadata_to_use = None
        else:
            metadata_to_use = metadata

        # Building the epochs
        epochs = mne.Epochs(
            raw, phase_events, event_id=available,
            tmin=epoch_tmin, tmax=epoch_tmax, baseline=baseline_corr,
            preload=True, reject_by_annotation=True, verbose=False,
            metadata=metadata_to_use,
        )

        # Sanity check: do trigger-derived and CSV-derived labels agree for the same epoch?
        if metadata_to_use is not None:
            # Inversion of keys and values of available: id_to_label now maps code to label
            id_to_label = {v: k for k, v in available.items()}
            # List comprehension: for every trigger code actually present in the final epochs object, 
            # look up its label. Result: a list of condition labels, in the same order as the epochs.
            triggered_labels = [id_to_label[code] for code in epochs.events[:, 2]]

            # Converts the dataframe column to a Python list, to allow comparison to triggered_labels
            beh_labels = epochs.metadata['condition_label'].tolist()

            # Create a list of indices, where the pair of (a) triggered_labels[i] and beh_labels [i] does not line up
            mismatches = [i for i, (a, b) in enumerate(zip(triggered_labels, beh_labels)) if a != b]
            if mismatches:
                log["warnings"].append(
                    f"{phase}: {len(mismatches)} trigger/CSV condition mismatches "
                    f"(first at epoch index {mismatches[0]}) — inspect trial order."
                )

        epochs_dict[phase] = epochs

    log["steps_completed"].append("epoching")
    log["n_epochs_pre_autoreject"] = {k: len(v) for k, v in epochs_dict.items()}
    # ==========================================
    # STEP 6: AUTOREJECT (separately per phase)
    # ==========================================
    # The Autoreject function uses cross-validation to identify an optimal peak-to-peak threshold for a given dataset
    # Based on it, bad channels are either interpolated or epochs are dropped
    epochs_clean = {}
    for phase, epochs in epochs_dict.items():
        # Evoked BEFORE autoreject
        fig_evoked_before = epochs.average().plot(show=False)
        save_fig(fig_evoked_before, plots_dir, f"02_evoked_{phase}_before_autoreject.png")

        ar = AutoReject(
            random_state=97,  # random seed
            n_jobs=1, # use 1 CPU core
            verbose=False
            ) 
        epochs_ar, reject_log = ar.fit_transform(epochs, return_log=True)
        epochs_clean[phase] = epochs_ar

        log.setdefault("n_epochs_post_autoreject", {})[phase] = len(epochs_ar)
        log.setdefault("autoreject_pct_dropped", {})[phase] = round(
            100 * (1 - len(epochs_ar) / len(epochs)), 1
        )

        # NEW: per-condition breakdown, same spot as the other post-AR counts
        log.setdefault("n_epochs_per_condition", {})[phase] = (
            epochs_ar.metadata['condition_label'].value_counts().to_dict()
            if epochs_ar.metadata is not None else {}
        )
        # Evoked AFTER autoreject
        fig_evoked_after = epochs_ar.average().plot(show=False)
        save_fig(fig_evoked_after, plots_dir, f"02_evoked_{phase}_after_autoreject.png")

        # Reject log visualization (which epochs/channels were dropped/interpolated)
        fig_reject = reject_log.plot(show=False)
        save_fig(fig_reject, plots_dir, f"05_autoreject_log_{phase}.png")

    
    log["steps_completed"].append("autoreject")


    # ==========================================
    # STEP 7: AVERAGE RE-REFERENCING
    # ==========================================
    for phase, epochs in epochs_clean.items():
        epochs.set_eeg_reference('average', projection=False, verbose=False)
    log["steps_completed"].append("average_reference")

    # ==========================================
    # STEP 8: SAVE OUTPUTS
    # ==========================================
    subj_deriv_dir.mkdir(parents=True, exist_ok=True)
    epochs_clean['encoding'].save(epochs_enc_path, overwrite=overwrite) if 'encoding' in epochs_clean else None
    epochs_clean['retrieval'].save(epochs_ret_path, overwrite=overwrite) if 'retrieval' in epochs_clean else None

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"sub-{subject_id}: preprocessing complete. "
          f"Encoding epochs: {log.get('n_epochs_post_autoreject', {}).get('encoding', 'N/A')}, "
          f"Retrieval epochs: {log.get('n_epochs_post_autoreject', {}).get('retrieval', 'N/A')}")

    return log

### 3. Run preprocessing function

#### 3.1 Single subject

Use this cell if you want to test the output of the preprocessing function for a single subject, e.g., to debug and troubleshoot.

In [24]:
test_log = preprocess_subject(
    subject_id="17",
    bad_channels=["P2"],
    bad_ics=[0, 1, 2, 3, 6, 7, 8, 10, 11, 14, 19, 22, 23, 25, 26, 36, 40],
    overwrite=True,
)


Reading 0 ... 3611599  =      0.000 ...  3611.599 secs...
Effective window size : 2.048 (s)
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\ica_qc\sub-17_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (61 components)
    Zeroing out 17 ICA components
    Projecting back using 62 PCA components
Dropped 2 epochs: 1, 2
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
sub-17: preprocessing complete. Encoding epochs: 62, Retrieval epochs: 84


#### 3.2 Batch preprocessing

Run the cells below if you want to run the pre-processing for all subjects that underwent the BIDS conversion and were visually inspected and configured in `2_manual_qc_and_config.ipynb`. Right now (July 22nd, 2026), this amounts to 18 participants.

The cell loads the configuration CSV file into a dataframe, which includes the subject numbers as an identifier, plus information about the integrity of the data recordings (EEG, behavioral) and importantly, the channels selected for interpolation aswell as the independent components to use for artefact correction.

In [3]:
# We loaded df_config in the set-up cell already

# Retrieve indices of subjects, that (1) are not behaviorally excluded &
# (2) have complete EEG recordings for encoding and retrieval phases

ready_mask = (
    (~df_config['is_excluded'].astype(bool)) &
    (df_config['eeg_enc_recorded'] == True) &
    (df_config['eeg_ret_recorded'] == True)
)
subjects_to_process = df_config[ready_mask]

print(f"{len(subjects_to_process)} of {len(df_config)} subjects ready for batch preprocessing.")

18 of 18 subjects ready for batch preprocessing.


This cell performs the actual preprocessing, looping over the selected participants.
Be careful! This can take a while to run. For me, about three minutes per participant. 

In [5]:
batch_summary = []
for _, row in subjects_to_process.iterrows():
    subj_id = row['subject']
    bad_chs = [c.strip() for c in row['bad_channels'].split(',')] if pd.notna(row['bad_channels']) and row['bad_channels'] else []
    bad_ic_list = [int(x.strip()) for x in row['bad_icas'].split(',')] if pd.notna(row['bad_icas']) and row['bad_icas'] else []

    try:
        log = preprocess_subject(
            subject_id=subj_id,
            bad_channels=bad_chs,
            bad_ics=bad_ic_list,
            overwrite=True,  # skip subjects already processed
        )
        batch_summary.append({"subject": subj_id, "status": "success", **log.get("n_epochs_post_autoreject", {})})
    except Exception as e:
        print(f"sub-{subj_id}: FAILED — {e}")
        batch_summary.append({"subject": subj_id, "status": "failed", "error": str(e)})

df_batch_summary = pd.DataFrame(batch_summary)
df_batch_summary.to_csv(derivatives_dir / "batch_preprocessing_summary.csv", index=False)
display(df_batch_summary)

Reading 0 ... 3611599  =      0.000 ...  3611.599 secs...
Effective window size : 2.048 (s)
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\ica_qc\sub-17_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (61 components)
    Zeroing out 17 ICA components
    Projecting back using 62 PCA components
Dropped 2 epochs: 1, 2
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
sub-17: preprocessing complete. Encoding epochs: 62, Retrieval epochs: 84
Reading 0 ... 3068039  =      0.000 ...  3068.039 secs...
Effective window size : 2.048 (s)
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
Reading c:\Users\noahm\proj

,subject,status,encoding,retrieval
0,17,success,62,84
1,20,success,64,80
2,21,success,44,51
3,22,success,61,84
4,23,success,64,84
5,26,success,64,81
6,27,success,63,74
7,30,success,62,83
8,32,success,64,84
9,33,success,64,82
